In [1]:
pip install opencv-python numpy matplotlib ipywidgets pillow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

save_folder = 'edited_photos'
os.makedirs(save_folder, exist_ok=True)

file_bytes = None
processed_img = None
user_filename = "photo.jpg"

upload_btn = widgets.FileUpload(accept='image/*', multiple=False, description='Upload Photo', button_style='primary')
filter_menu = widgets.Dropdown(
    options=["Original", "Red", "Blue", "Green", "Brighten", "Gray", "Purple", "Neon"],
    value="Original",
    layout={'width': '140px'}
)
apply_btn = widgets.Button(description='Apply Filter', button_style='success')
download_btn = widgets.Button(description='Get Link', button_style='info')

image_viewer = widgets.Output()
link_viewer = widgets.Output()

def process_filter(original_image, filter_name):
    img = original_image.copy()
    if filter_name == "Brighten":
        return cv2.convertScaleAbs(img, alpha=1.4, beta=25)
    elif filter_name == "Gray":
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    elif filter_name == "Red":
        img[:, :, 0] = 0  
        img[:, :, 1] = 0  
        return img
    elif filter_name == "Blue":
        img[:, :, 1] = 0  
        img[:, :, 2] = 0  
        return img
    elif filter_name == "Green":
        img[:, :, 0] = 0  
        img[:, :, 2] = 0  
        return img
    elif filter_name == "Purple":
        img[:, :, 1] = 0  
        return img
    elif filter_name == "Neon":
        smooth = cv2.bilateralFilter(img, 9, 75, 75)
        edges = cv2.Laplacian(smooth, cv2.CV_8U, ksize=5)
        return cv2.bitwise_not(edges)
    return img

def check_upload(change):
    global file_bytes, user_filename
    if change['new']:
        link_viewer.clear_output()
        uploaded_data = change['new'][0]
        file_bytes = uploaded_data['content']
        user_filename = uploaded_data['name']
        filter_menu.value = "Original"
        update_studio_display(None)

def update_studio_display(button_click):
    global processed_img
    if file_bytes is None:
        with image_viewer:
            print("Please upload a photo first!")
        return
    link_viewer.clear_output()
    
    with image_viewer:
        clear_output(wait=True)
        image_array = np.frombuffer(file_bytes, np.uint8)
        original_bgr = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
        
        chosen_filter = filter_menu.value
        processed_img = process_filter(original_bgr, chosen_filter)
        
        left_side = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
        right_side = cv2.cvtColor(processed_img, cv2.COLOR_BGR2RGB)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        axes[0].imshow(left_side)
        axes[0].set_title("Original Image", fontsize=11, fontweight='bold')
        axes[0].axis('off')
        
        axes[1].imshow(right_side)
        axes[1].set_title(f"Filter: {chosen_filter}", fontsize=11, fontweight='bold')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()

def build_download_link(button_click):
    if processed_img is None:
        with link_viewer:
            print("Apply a filter first.")
        return
        
    link_viewer.clear_output()
    with link_viewer:
        name, extension = os.path.splitext(user_filename)
        filter_tag = filter_menu.value.lower().replace(" ", "_")
        
        new_name = f"{name}_{filter_tag}{extension}"
        full_save_path = os.path.join(save_folder, new_name)
        
        # Fixed variable path bug here
        cv2.imwrite(full_save_path, processed_img)
        display(FileLink(full_save_path, result_html_prefix="Click to Download: "))

upload_btn.observe(check_upload, names='value')
apply_btn.on_click(update_studio_display)
download_btn.on_click(build_download_link)

control_box = widgets.HBox(
    [filter_menu, apply_btn, download_btn, link_viewer], 
    layout={
        'border': '1px solid #cccccc', 
        'padding': '10px', 
        'align_items': 'center',
        'width': '100%',
        'justify_content': 'flex-start'
    }
)

minimal_studio = widgets.VBox([
    widgets.HBox([upload_btn], layout={'justify_content': 'center', 'padding': '10px'}),
    image_viewer,
    control_box
])

display(minimal_studio)